In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path('.').resolve().parent / 'data'

train_transaction = pd.read_csv(data_dir / 'train_transaction.csv')
train_identity = pd.read_csv(data_dir / 'train_identity.csv')
df = train_transaction.merge(train_identity, on='TransactionID', how='left')

df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [2]:
col_info = pd.DataFrame({
    'column_name': df.columns,
    'data_type': df.dtypes.values,
    'missing_%': df.isnull().mean().values * 100,
    'n_unique': df.nunique().values
})

col_info = col_info.sort_values(by='missing_%', ascending=False)

col_info.head(20)

,column_name,data_type,missing_%,n_unique
417,id_24,float64,99.196159,12
418,id_25,float64,99.130965,341
400,id_07,float64,99.127070,84
401,id_08,float64,99.127070,94
414,id_21,float64,99.126393,490
419,id_26,float64,99.125715,95
420,id_27,str,99.124699,2
416,id_23,str,99.124699,3
415,id_22,float64,99.124699,25
14,dist2,float64,93.628374,1751


In [3]:
col_info

,column_name,data_type,missing_%,n_unique
417,id_24,float64,99.196159,12
418,id_25,float64,99.130965,341
400,id_07,float64,99.127070,84
401,id_08,float64,99.127070,94
414,id_21,float64,99.126393,490
...,...,...,...,...
27,C11,float64,0.000000,1476
30,C14,float64,0.000000,1108
29,C13,float64,0.000000,1597
28,C12,float64,0.000000,1199


In [ ]:
col_info.to_csv(data_dir / 'column_summary.csv', index=False)

In [5]:
# Drop columns with >90% missing
missing_ratio = df.isnull().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.9].index

df.drop(columns=high_missing_cols, inplace=True)

print("Dropped high-missing:", len(high_missing_cols))

Dropped high-missing: 12


In [6]:
# Drop columns with only 1 unique value
low_var_cols = [col for col in df.columns if df[col].nunique() <= 1]

df.drop(columns=low_var_cols, inplace=True)

print("Dropped low variance:", len(low_var_cols))

Dropped low variance: 0


In [7]:
df['TransactionDT'] = df['TransactionDT'] / 3600

df = df.sort_values(['card1', 'TransactionDT'])

df['tx_count'] = df.groupby('card1').cumcount()
df['tx_amt_mean_5'] = df.groupby('card1')['TransactionAmt'].transform(lambda x: x.rolling(5).mean())
df['tx_amt_std_5'] = df.groupby('card1')['TransactionAmt'].transform(lambda x: x.rolling(5).std())

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2414709620.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['tx_count'] = df.groupby('card1').cumcount()
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2414709620.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['tx_amt_mean_5'] = df.groupby('card1')['TransactionAmt'].transform(lambda x: x.rolling(5).mean())
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2414709620.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `

In [8]:
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

df['user_mean_amt'] = df.groupby('card1')['TransactionAmt'].transform('mean')
df['amt_deviation'] = df['TransactionAmt'] - df['user_mean_amt']

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3376446197.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3376446197.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['user_mean_amt'] = df.groupby('card1')['TransactionAmt'].transform('mean')
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3376446197.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

In [9]:
df['hour'] = df['TransactionDT'] % 24
df['day'] = df['TransactionDT'] // 24

df['tx_per_day'] = df.groupby(['card1', 'day'])['TransactionID'].transform('count')

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2291394030.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hour'] = df['TransactionDT'] % 24
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2291394030.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['day'] = df['TransactionDT'] // 24
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2291394030.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

In [10]:
# Shared email
if 'P_emaildomain' in df.columns:
    email_users = df.groupby('P_emaildomain')['card1'].nunique()
    df['email_user_count'] = df['P_emaildomain'].map(email_users)

# Shared device
if 'DeviceType' in df.columns:
    device_users = df.groupby('DeviceType')['card1'].nunique()
    df['device_user_count'] = df['DeviceType'].map(device_users)

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3721693560.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['email_user_count'] = df['P_emaildomain'].map(email_users)
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3721693560.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['device_user_count'] = df['DeviceType'].map(device_users)


In [11]:
if 'DeviceType' in df.columns:
    df['device_count'] = df.groupby('card1')['DeviceType'].transform('nunique')

if 'P_emaildomain' in df.columns:
    df['email_count'] = df.groupby('card1')['P_emaildomain'].transform('nunique')

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\592321820.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['device_count'] = df.groupby('card1')['DeviceType'].transform('nunique')
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\592321820.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['email_count'] = df.groupby('card1')['P_emaildomain'].transform('nunique')


In [12]:
high_card_cols = ['card1', 'card2', 'addr1', 'P_emaildomain']

for col in high_card_cols:
    if col in df.columns:
        freq = df[col].value_counts(dropna=False)
        df[col + '_freq'] = df[col].map(freq)

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\872208917.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + '_freq'] = df[col].map(freq)


In [13]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include=['object', 'string']).columns

for col in cat_cols:
    df[col] = df[col].astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [14]:
df.replace([np.inf, -np.inf], -999, inplace=True)
df.fillna(-999, inplace=True)

print("Final shape:", df.shape)

Final shape: (590540, 439)


In [ ]:
df.to_csv(data_dir / 'processed_fraud_data.csv', index=False)